# EXIST 2026 Subtask 2.1: Advanced Dual-Encoder Implementation
This notebook implements the advanced dual-encoder architecture with hardware optimization, multi-task learning for sensor data, and cross-attention fusion.


## 1. Imports and Configuration


In [1]:
import os
import json
from pathlib import Path
import numpy as np
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import AutoModel, AutoTokenizer, AutoProcessor, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from collections import defaultdict
from accelerate import Accelerator

DEFAULT_DATASET_NAME = "EXIST 2026 Memes Dataset"
DEFAULT_REPO_NAME = "exist26"

def build_data_root_candidates(project_root):
    candidates = [
        Path("/kaggle/input/datasets/thirumuruganra/exist-2026-memes"),
        Path("/kaggle/input/exist-2026-memes"),
        Path("/kaggle/input"),
        project_root,
        project_root / DEFAULT_REPO_NAME,
        project_root.parent,
        project_root.parent / DEFAULT_REPO_NAME,
        Path("/content"),
        Path("/content") / DEFAULT_REPO_NAME,
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive") / DEFAULT_REPO_NAME,
        Path("/content/drive/MyDrive/datasets"),
        Path("/content/drive/MyDrive/Colab Notebooks"),
    ]
    
    unique_candidates = []
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.exists() else candidate
        key = str(resolved)
        if key not in seen:
            unique_candidates.append(candidate)
            seen.add(key)
    return unique_candidates

def resolve_data_root(candidates):
    dataset_roots = []
    for candidate in candidates:
        dataset_roots.append(candidate)
        dataset_roots.append(candidate / DEFAULT_DATASET_NAME)
    
    for dataset_root in dataset_roots:
        training_json = dataset_root / "training" / "EXIST2026_training.json"
        test_json = dataset_root / "test" / "EXIST2026_test_clean.json"
        if training_json.exists() and test_json.exists():
            return dataset_root
    
    if Path("/kaggle/input").exists():
        for training_json in Path("/kaggle/input").glob("**/EXIST2026_training.json"):
            dataset_root = training_json.parent.parent
            test_json = dataset_root / "test" / "EXIST2026_test_clean.json"
            if test_json.exists():
                return dataset_root
    
    checked_paths = "\n".join(str(path) for path in dataset_roots)
    raise FileNotFoundError(
        "Could not locate the EXIST 2026 dataset. Update the dataset path candidates with your dataset location.\n"
        f"Checked:\n{checked_paths}"
    )

class Config:
    PROJECT_ROOT = Path.cwd().resolve()
    DATA_ROOT_CANDIDATES = build_data_root_candidates(PROJECT_ROOT)
    DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
    TRAIN_BASE = DATA_ROOT / "training"
    TEST_BASE = DATA_ROOT / "test"
    
    TRAIN_JSON = str(TRAIN_BASE / "EXIST2026_training.json")
    TEST_JSON = str(TEST_BASE / "EXIST2026_test_clean.json")
    TRAIN_IMAGE_DIR = str(TRAIN_BASE / "memes")
    TEST_IMAGE_DIR = str(TEST_BASE / "memes")
    
    OUTPUT_DIR_CANDIDATES = [
        Path("/kaggle/working"),
        Path("/content"),
        PROJECT_ROOT,
    ]
    OUTPUT_DIR = next((path for path in OUTPUT_DIR_CANDIDATES if path.exists()), PROJECT_ROOT)
    SOFT_OUTPUT_JSON = str(OUTPUT_DIR / "soft_submission.json")
    HARD_OUTPUT_JSON = str(OUTPUT_DIR / "hard_submission.json")
    
    TEXT_MODEL = "cardiffnlp/twitter-xlm-roberta-base"
    VISION_MODEL = "google/siglip-base-patch16-256"
    TEXT_MAX_LENGTH = 128
    TEXT_TRAINABLE_LAYERS = 12
    FREEZE_VISION_ENCODER = False
    
    BATCH_SIZE = 16
    EPOCHS = 5
    TEXT_ENCODER_LR = 1e-5
    VISION_ENCODER_LR = 5e-6
    HEAD_LR = 5e-5
    WARMUP_RATIO = 0.1
    VAL_SPLIT = 0.2
    SEED = 42
    DROPOUT_RATE = 0.3
    WEIGHT_DECAY = 0.01
    NUM_WORKERS = 2
    PIN_MEMORY = True
    EARLY_STOPPING_PATIENCE = 2
    BEST_MODEL_PATH = str(OUTPUT_DIR / "best_model.pth")
    THRESHOLD_GRID = np.linspace(0.30, 0.70, 9)
    
    ALPHA = 0.0  # Completely disables the auxiliary multi-task sensor MSE loss



## 2. Utility Functions and Dataset



In [2]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_soft_label(labels):
    if not labels:
        return 0.0
    return float(np.mean([1.0 if label == "YES" else 0.0 for label in labels]))

def build_stratify_label(item_id, item_data):
    soft_yes = get_soft_label(item_data.get("labels_task2_1", []))
    if soft_yes <= 0.20:
        agreement_bucket = "mostly_no"
    elif soft_yes >= 0.80:
        agreement_bucket = "mostly_yes"
    else:
        agreement_bucket = "mixed"
    language_bucket = "es" if str(item_id).startswith("310") else "en"
    return f"{language_bucket}_{agreement_bucket}"

def masked_mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def compute_soft_bce(predictions, targets):
    total_loss = 0.0
    for exist_id in predictions:
        pred_soft_yes = float(np.mean(predictions[exist_id]))
        true_soft_yes = float(np.mean(targets[exist_id]))
        pred_soft_yes = np.clip(pred_soft_yes, 1e-7, 1.0 - 1e-7)
        total_loss += -(
            true_soft_yes * np.log(pred_soft_yes)
            + (1.0 - true_soft_yes) * np.log(1.0 - pred_soft_yes)
        )
    return total_loss / max(len(predictions), 1)

def compute_binary_metrics(predictions, targets, threshold):
    tp = fp = tn = fn = 0
    for exist_id in predictions:
        pred_yes = float(np.mean(predictions[exist_id]))
        actual_yes = float(np.mean(targets[exist_id]))
        pred_hard = 1 if pred_yes >= threshold else 0
        actual_hard = 1 if actual_yes >= 0.5 else 0
        
        if actual_hard == 1 and pred_hard == 1:
            tp += 1
        elif actual_hard == 0 and pred_hard == 0:
            tn += 1
        elif actual_hard == 0 and pred_hard == 1:
            fp += 1
        else:
            fn += 1
            
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn, "precision": precision, "recall": recall, "f1": f1, "accuracy": accuracy}

def find_best_threshold(predictions, targets, thresholds):
    best_threshold = 0.5
    best_metrics = compute_binary_metrics(predictions, targets, best_threshold)
    for threshold in thresholds:
        metrics = compute_binary_metrics(predictions, targets, float(threshold))
        if metrics["f1"] > best_metrics["f1"]:
            best_threshold = float(threshold)
            best_metrics = metrics
    return best_threshold, best_metrics

class ExistMultimodalDataset(Dataset):
    def __init__(self, json_path, image_dir, text_tokenizer, vision_processor, is_test=False, allowed_ids=None, sensor_keys=None, sensor_stats=None):
        self.image_dir = image_dir
        self.text_tokenizer = text_tokenizer
        self.vision_processor = vision_processor
        self.is_test = is_test
        
        with open(json_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            
        self.samples = []
        self.sensor_keys = sensor_keys
        
        if self.sensor_keys is None:
            sensor_keys_set = set()
            for item_data in raw_data.values():
                if "sensorial" in item_data:
                    for mod in item_data["sensorial"].get("modalities", {}).values():
                        for user_data in mod.get("by_user", {}).values():
                            sensor_keys_set.update(user_data.keys())
            self.sensor_keys = sorted(list(sensor_keys_set))
            
        self.sensor_key_to_idx = {k: i for i, k in enumerate(self.sensor_keys)}
        self.sensor_dim = len(self.sensor_keys)
        
        for item_id, item_data in raw_data.items():
            if allowed_ids is not None and item_id not in allowed_ids:
                continue
            
            sample = {
                "id_EXIST": item_id,
                "text": item_data.get("text", ""),
                "img_path": os.path.join(image_dir, item_data.get("meme", "")),
                "label": 0.0 if is_test else get_soft_label(item_data.get("labels_task2_1", []))
            }
            
            sensor_vector = np.zeros(self.sensor_dim, dtype=np.float32)
            sensor_mask = 0.0
            feature_valid = np.zeros(self.sensor_dim, dtype=bool)
            
            if "sensorial" in item_data and self.sensor_dim > 0:
                counts = np.zeros(self.sensor_dim, dtype=np.float32)
                for mod in item_data["sensorial"].get("modalities", {}).values():
                    for user_data in mod.get("by_user", {}).values():
                        for k, v in user_data.items():
                            if v is not None and k in self.sensor_key_to_idx:
                                idx = self.sensor_key_to_idx[k]
                                sensor_vector[idx] += v
                                counts[idx] += 1
                valid = counts > 0
                if np.any(valid):
                    sensor_vector[valid] /= counts[valid]
                    sensor_mask = 1.0
                    feature_valid = valid
                    
            sample["sensor_vector"] = sensor_vector
            sample["sensor_mask"] = sensor_mask
            sample["feature_valid"] = feature_valid
            self.samples.append(sample)
            
        if sensor_stats is None:
            self.sensor_stats = {'mean': np.zeros(self.sensor_dim, dtype=np.float32), 
                                 'std': np.ones(self.sensor_dim, dtype=np.float32)}
            for i in range(self.sensor_dim):
                feature_vals = [s["sensor_vector"][i] for s in self.samples if s["feature_valid"][i]]
                if feature_vals:
                    self.sensor_stats['mean'][i] = np.mean(feature_vals)
                    self.sensor_stats['std'][i] = np.std(feature_vals)
        else:
            self.sensor_stats = sensor_stats

        for s in self.samples:
            if s["sensor_mask"] == 1.0:
                valid = s["feature_valid"]
                s["sensor_vector"][valid] = (s["sensor_vector"][valid] - self.sensor_stats['mean'][valid]) / (self.sensor_stats['std'][valid] + 1e-8)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        text_inputs = self.text_tokenizer(
            sample["text"],
            padding='max_length',
            truncation=True,
            max_length=Config.TEXT_MAX_LENGTH,
            return_tensors="pt"
        )
        
        item = {
            "id_EXIST": sample["id_EXIST"],
            "input_ids": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "label": torch.tensor(sample["label"], dtype=torch.float),
            "sensor_vector": torch.tensor(sample["sensor_vector"], dtype=torch.float),
            "sensor_mask": torch.tensor(sample["sensor_mask"], dtype=torch.float)
        }
        
        try:
            raw_image = Image.open(sample["img_path"])
            image = raw_image.convert("RGBA").convert("RGB")
        except Exception:
            image = Image.new('RGB', (224, 224), color='black')
            
        vision_inputs = self.vision_processor(images=image, return_tensors="pt")
        item["pixel_values"] = vision_inputs["pixel_values"].squeeze(0)
        
        return item



## 3. Model Architecture



In [3]:
def set_trainable_text_layers(text_model, trainable_layers):
    for param in text_model.parameters():
        param.requires_grad = False
    
    embeddings = getattr(text_model, "embeddings", None)
    if embeddings is not None:
        for param in embeddings.parameters():
            param.requires_grad = True
    
    encoder = getattr(text_model, "encoder", None)
    if encoder is not None and hasattr(encoder, "layer"):
        total_layers = len(encoder.layer)
        if trainable_layers >= total_layers:
            layers = encoder.layer
        elif trainable_layers > 0:
            layers = encoder.layer[-trainable_layers:]
        else:
            layers = []
        for layer in layers:
            for param in layer.parameters():
                param.requires_grad = True
    
    pooler = getattr(text_model, "pooler", None)
    if pooler is not None:
        for param in pooler.parameters():
            param.requires_grad = True

class CrossAttentionFusionModel(nn.Module):
    def __init__(self, config, sensor_dim):
        super().__init__()
        self.text_model = AutoModel.from_pretrained(config.TEXT_MODEL)
        self.vision_model = AutoModel.from_pretrained(config.VISION_MODEL).vision_model
        
        set_trainable_text_layers(self.text_model, config.TEXT_TRAINABLE_LAYERS)
        if config.FREEZE_VISION_ENCODER:
            for param in self.vision_model.parameters():
                param.requires_grad = False
                
        self.text_model.gradient_checkpointing_enable()
        self.vision_model.gradient_checkpointing_enable()
        
        text_dim = self.text_model.config.hidden_size
        vision_dim = self.vision_model.config.hidden_size
        fused_dim = text_dim
        
        self.vision_proj = nn.Linear(vision_dim, text_dim) if vision_dim != text_dim else nn.Identity()
        self.pre_fusion_dropout = nn.Dropout(p=0.1)
        self.ln_text = nn.LayerNorm(text_dim)
        self.ln_vision = nn.LayerNorm(text_dim)
        self.cross_attn = nn.MultiheadAttention(embed_dim=text_dim, num_heads=8, batch_first=True)
        
        self.ln_ffn = nn.LayerNorm(text_dim)
        self.ffn = nn.Sequential(
            nn.Linear(text_dim, 4 * text_dim),
            nn.GELU(),
            nn.Linear(4 * text_dim, text_dim)
        )
        
        self.fusion_norm = nn.LayerNorm(fused_dim)
        self.fusion_act = nn.GELU()
        self.classifier = nn.Linear(fused_dim, 1)
        
        self.sensor_head = nn.Sequential(
            nn.Linear(text_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(config.DROPOUT_RATE),
            nn.Linear(256, sensor_dim)
        )

    def forward(self, input_ids, attention_mask, pixel_values):
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_seq = text_outputs.last_hidden_state
        
        vision_outputs = self.vision_model(pixel_values=pixel_values)
        vision_seq = vision_outputs.last_hidden_state
        
        vision_seq = self.vision_proj(vision_seq)
        text_seq = self.pre_fusion_dropout(text_seq)
        vision_seq = self.pre_fusion_dropout(vision_seq)
        
        text_seq_norm = self.ln_text(text_seq)
        vision_seq_norm = self.ln_vision(vision_seq)
        
        attn_output, _ = self.cross_attn(
            query=text_seq_norm,
            key=vision_seq_norm,
            value=vision_seq_norm
        )
        
        text_seq = text_seq + attn_output
        text_seq = text_seq + self.ffn(self.ln_ffn(text_seq))
        
        fused = masked_mean_pool(text_seq, attention_mask)
        fused = self.fusion_norm(fused)
        fused = self.fusion_act(fused)
        class_logits = self.classifier(fused).squeeze(-1)
        sensor_preds = self.sensor_head(fused)
        
        return class_logits, sensor_preds

def build_optimizer(model, config):
    text_params = []
    vision_params = []
    head_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("text_model."):
            text_params.append(param)
        elif name.startswith("vision_model."):
            vision_params.append(param)
        else:
            head_params.append(param)
            
    parameter_groups = []
    if text_params:
        parameter_groups.append({"params": text_params, "lr": config.TEXT_ENCODER_LR, "weight_decay": config.WEIGHT_DECAY})
    if vision_params:
        parameter_groups.append({"params": vision_params, "lr": config.VISION_ENCODER_LR, "weight_decay": config.WEIGHT_DECAY})
    if head_params:
        parameter_groups.append({"params": head_params, "lr": config.HEAD_LR, "weight_decay": config.WEIGHT_DECAY})
        
    return torch.optim.AdamW(parameter_groups)



## 4. Training Loop



In [4]:
def train_model():
    seed_everything(Config.SEED)
    
    # Initialize Accelerator with fp16
    accelerator = Accelerator(mixed_precision="fp16")
    
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    processor = AutoProcessor.from_pretrained(Config.VISION_MODEL)
    
    with open(Config.TRAIN_JSON, 'r', encoding='utf-8') as f:
        full_data = json.load(f)
        
    all_meme_ids = list(full_data.keys())
    stratify_labels = [build_stratify_label(meme_id, full_data[meme_id]) for meme_id in all_meme_ids]
    
    try:
        train_ids_list, val_ids_list = train_test_split(all_meme_ids, test_size=Config.VAL_SPLIT, random_state=Config.SEED, shuffle=True, stratify=stratify_labels)
    except ValueError:
        train_ids_list, val_ids_list = train_test_split(all_meme_ids, test_size=Config.VAL_SPLIT, random_state=Config.SEED, shuffle=True)
        
    train_ids = set(train_ids_list)
    val_ids = set(val_ids_list)
    
    train_dataset = ExistMultimodalDataset(Config.TRAIN_JSON, Config.TRAIN_IMAGE_DIR, tokenizer, processor, allowed_ids=train_ids)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=Config.NUM_WORKERS, pin_memory=Config.PIN_MEMORY)
    
    val_dataset = ExistMultimodalDataset(Config.TRAIN_JSON, Config.TRAIN_IMAGE_DIR, tokenizer, processor, allowed_ids=val_ids, sensor_keys=train_dataset.sensor_keys, sensor_stats=train_dataset.sensor_stats)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=Config.PIN_MEMORY)
    
    model = CrossAttentionFusionModel(Config, sensor_dim=train_dataset.sensor_dim)
    optimizer = build_optimizer(model, Config)
    
    # Prepare with accelerate
    model, optimizer, train_loader, val_loader = accelerator.prepare(model, optimizer, train_loader, val_loader)
    
    criterion_primary = nn.BCEWithLogitsLoss()
    criterion_aux = nn.MSELoss(reduction='none')
    
    total_steps = len(train_loader) * Config.EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * Config.WARMUP_RATIO), num_training_steps=total_steps)
    
    epoch_history = []
    best_val_loss = float('inf')
    best_threshold = 0.5
    best_metrics = None
    best_val_preds = None
    best_val_targets = None
    epochs_without_improvement = 0
    
    for epoch in range(Config.EPOCHS):
        model.train()
        total_train_loss = 0.0
        
        for batch in train_loader:
            optimizer.zero_grad()
            
            logits, sensor_preds = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                pixel_values=batch["pixel_values"]
            )
            
            primary_loss = criterion_primary(logits, batch["label"])
            
            aux_loss = criterion_aux(sensor_preds, batch["sensor_vector"])
            # average over sensor dimension and then multiply by mask
            aux_loss = (aux_loss.mean(dim=1) * batch["sensor_mask"]).mean()
            
            loss = primary_loss + Config.ALPHA * aux_loss
            
            accelerator.backward(loss)
            optimizer.step()
            scheduler.step()
            total_train_loss += loss.item()
            
        model.eval()
        val_preds = defaultdict(list)
        val_targets = defaultdict(list)
        
        with torch.no_grad():
            for batch in val_loader:
                logits, _ = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    pixel_values=batch["pixel_values"]
                )
                
                logits = accelerator.gather_for_metrics(logits)
                labels = accelerator.gather_for_metrics(batch["label"])
                ids = accelerator.gather_for_metrics(batch["id_EXIST"])
                
                probs = torch.sigmoid(logits).cpu().numpy()
                labels = labels.cpu().numpy()
                
                for exist_id, prob, label in zip(ids, probs, labels):
                    val_preds[exist_id].append(float(prob))
                    val_targets[exist_id].append(float(label))
                    
        avg_val_loss = compute_soft_bce(val_preds, val_targets)
        epoch_threshold, epoch_metrics = find_best_threshold(val_preds, val_targets, Config.THRESHOLD_GRID)
        
        save_msg = ""
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_threshold = epoch_threshold
            best_metrics = epoch_metrics
            best_val_preds = {exist_id: probs[:] for exist_id, probs in val_preds.items()}
            best_val_targets = {exist_id: labels[:] for exist_id, labels in val_targets.items()}
            
            epochs_without_improvement = 0
            save_msg = " <-- BEST MODEL SAVED"
            if accelerator.is_main_process:
                unwrapped_model = accelerator.unwrap_model(model)
                accelerator.save(unwrapped_model.state_dict(), Config.BEST_MODEL_PATH)
        else:
            epochs_without_improvement += 1
            
        accelerator.wait_for_everyone()
            
        if accelerator.is_main_process:
            log_line = (
                f"Epoch {epoch+1}/{Config.EPOCHS} | "
                f"Train Loss: {total_train_loss / max(len(train_loader), 1):.4f} | "
                f"Validation Soft-BCE Loss: {avg_val_loss:.4f} | "
                f"Threshold: {epoch_threshold:.2f} | "
                f"F1: {epoch_metrics['f1']:.4f}{save_msg}"
            )
            print(log_line)
            epoch_history.append(log_line)
            
        if epochs_without_improvement >= Config.EARLY_STOPPING_PATIENCE:
            if accelerator.is_main_process:
                print(f"Early stopping triggered after {epoch + 1} epochs.")
            break
            
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        print(f"\nTraining complete. Reloading the best weights from disk...")
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.load_state_dict(torch.load(Config.BEST_MODEL_PATH, map_location="cpu"))
        
    artifacts = {
        "tokenizer": tokenizer,
        "processor": processor,
        "sensor_keys": train_dataset.sensor_keys,
        "sensor_stats": train_dataset.sensor_stats
    }
    
    return accelerator.unwrap_model(model), artifacts, best_val_preds, best_val_targets, epoch_history, best_threshold, best_metrics



## 5. Inference and Execution



In [5]:
def rescale_probability(p, threshold=0.30):
    """Stretches probabilities piece-wise so the custom threshold maps cleanly to 0.50"""
    p = float(p)
    if p < threshold:
        # Scale [0.0, threshold] -> [0.0, 0.5]
        return 0.5 * (p / threshold)
    else:
        # Scale [threshold, 1.0] -> [0.5, 1.0]
        return 0.5 + 0.5 * ((p - threshold) / (1.0 - threshold))


def run_inference(model, tokenizer, processor, sensor_keys, hard_threshold=0.5):
    test_dataset = ExistMultimodalDataset(
        Config.TEST_JSON,
        Config.TEST_IMAGE_DIR,
        tokenizer,
        processor,
        is_test=True,
        sensor_keys=sensor_keys
    )
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS)
    
    accelerator = Accelerator(mixed_precision="fp16")
    model, test_loader = accelerator.prepare(model, test_loader)
    
    model.eval()
    meme_predictions = defaultdict(list)
    with torch.no_grad():
        for batch in test_loader:
            logits, _ = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                pixel_values=batch["pixel_values"]
            )
            
            logits = accelerator.gather(logits)
            ids = accelerator.gather(batch["id_EXIST"])
            probs = torch.sigmoid(logits).cpu().numpy()
            
            for exist_id, prob in zip(ids, probs):
                meme_predictions[exist_id].append(float(prob))
                
    if accelerator.is_main_process:
        soft_submission = []
        hard_submission = []
        for exist_id in sorted(meme_predictions, key=lambda x: str(x)):
            mean_yes = float(np.mean(meme_predictions[exist_id]))
            scaled_yes_prob = rescale_probability(mean_yes, threshold=hard_threshold)
            scaled_no_prob = 1.0 - scaled_yes_prob
            soft_submission.append({
                "test_case": "EXIST2025",
                "id": str(exist_id),
                "value": {"YES": scaled_yes_prob, "NO": scaled_no_prob}
            })
            hard_submission.append({
                "test_case": "EXIST2025",
                "id": str(exist_id),
                "value": "YES" if mean_yes >= hard_threshold else "NO"
            })
            
        with open(Config.SOFT_OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(soft_submission, f, indent=4)
        print(f"Soft predictions saved to {Config.SOFT_OUTPUT_JSON}")
            
        with open(Config.HARD_OUTPUT_JSON, 'w', encoding='utf-8') as f:
            json.dump(hard_submission, f, indent=4)
        print(f"Hard predictions saved to {Config.HARD_OUTPUT_JSON}")
            
    return

if __name__ == "__main__":
    print("Starting advanced dual-encoder training pipeline...")
    trained_model, artifacts, val_preds, val_targets, history, best_thresh, metrics = train_model()
    
    if val_preds is not None:
        print("\n--- Validation Set Results ---")
        print(f"Best Threshold: {best_thresh:.2f}")
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"F1 Score:  {metrics['f1']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print("------------------------------\n")
        print("\n--- PyEvALL Official Validation Metrics ---")
        val_soft_preds_list = []


        val_hard_preds_list = []
        val_soft_gold_list = []
        val_hard_gold_list = []
        
        for exist_id in sorted(val_preds.keys(), key=lambda x: str(x)):
            pred_mean_yes = float(np.mean(val_preds[exist_id]))
            scaled_yes_prob = rescale_probability(pred_mean_yes, threshold=best_thresh)
            scaled_no_prob = 1.0 - scaled_yes_prob
            target_mean_yes = float(np.mean(val_targets[exist_id]))
            target_mean_no = 1.0 - target_mean_yes
            
            val_soft_preds_list.append({
                "test_case": "EXIST2025",
                "id": str(exist_id),
                "value": {"YES": scaled_yes_prob, "NO": scaled_no_prob}
            })
            val_soft_gold_list.append({
                "test_case": "EXIST2025",
                "id": str(exist_id),
                "value": {"YES": target_mean_yes, "NO": target_mean_no}
            })
            
            val_hard_preds_list.append({
                "test_case": "EXIST2025",
                "id": str(exist_id),
                "value": "YES" if pred_mean_yes >= best_thresh else "NO"
            })
            val_hard_gold_list.append({
                "test_case": "EXIST2025",
                "id": str(exist_id),
                "value": "YES" if target_mean_yes >= 0.5 else "NO"
            })
            
        soft_preds_path = str(Config.OUTPUT_DIR / "val_soft_preds.json")
        soft_gold_path = str(Config.OUTPUT_DIR / "val_soft_gold.json")
        hard_preds_path = str(Config.OUTPUT_DIR / "val_hard_preds.json")
        hard_gold_path = str(Config.OUTPUT_DIR / "val_hard_gold.json")
            
        with open(soft_preds_path, 'w') as f: json.dump(val_soft_preds_list, f)
        with open(soft_gold_path, 'w') as f: json.dump(val_soft_gold_list, f)
        with open(hard_preds_path, 'w') as f: json.dump(val_hard_preds_list, f)
        with open(hard_gold_path, 'w') as f: json.dump(val_hard_gold_list, f)
            
        print("------------------------------\n")
        print("\nRunning inference on test set...")
        run_inference(trained_model, artifacts["tokenizer"], artifacts["processor"], artifacts["sensor_keys"], best_thresh)

Starting advanced dual-encoder training pipeline...


config.json:   0%|          | 0.00/652 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch 1/5 | Train Loss: 0.6740 | Validation Soft-BCE Loss: 0.6478 | Threshold: 0.45 | F1: 0.8051 <-- BEST MODEL SAVED
Epoch 2/5 | Train Loss: 0.5977 | Validation Soft-BCE Loss: 0.6463 | Threshold: 0.40 | F1: 0.8120 <-- BEST MODEL SAVED
Epoch 3/5 | Train Loss: 0.5087 | Validation Soft-BCE Loss: 0.6587 | Threshold: 0.30 | F1: 0.8146
Epoch 4/5 | Train Loss: 0.4728 | Validation Soft-BCE Loss: 0.6624 | Threshold: 0.30 | F1: 0.8069
Early stopping triggered after 4 epochs.

Training complete. Reloading the best weights from disk...

--- Validation Set Results ---
Best Threshold: 0.40
Accuracy:  0.7164
F1 Score:  0.8120
Precision: 0.7135
Recall:    0.9421
------------------------------


--- PyEvALL Official Validation Metrics ---
------------------------------


Running inference on test set...
Soft predictions saved to /kaggle/working/soft_submission.json
Hard predictions saved to /kaggle/working/hard_submission.json
